# Kapitel 19.4 - WSGI Grundlagen und Mini-Framework

Ab hier arbeiten wir mit WSGI, dem Standard-Interface zwischen Python-Webanwendung und Webserver.
Dieses Notebook ist absichtlich umfangreich und fuehrt dich von der Signatur bis zu einem Mini-Framework.

# Lernziele

Nach diesem Notebook kannst du:

- die WSGI-Schnittstelle fachlich korrekt erklaeren
- `environ` gezielt auswerten
- korrekte Statuszeilen und Header setzen
- Bytes-Rueckgaben sauber erzeugen
- einen kleinen Router fuer mehrere Endpunkte implementieren
- den Kern eines Mini-Frameworks nachvollziehen

# Voraussetzungen

Empfohlen:

- Kapitel 19.1 bis 19.3
- sichere Python-Funktionsnutzung
- gute Grundlagen in Dictionaries und String-Verarbeitung

Optional: Kenntnis von Flask ist hilfreich, aber nicht notwendig.

# Theorie

## WSGI in einem Satz

WSGI definiert, wie ein Python-Callable HTTP-Anfragen entgegennimmt und Antworten zurueckgibt.

## Standard-Signatur

`application(environ, start_response)`

- `environ`: Request-Metadaten
- `start_response`: Callback fuer Status + Header
- Rueckgabe: iterierbar mit Bytes

## Warum WSGI so wichtig ist

Frameworks und Server bleiben austauschbar, solange beide WSGI sprechen.
Das ist ein zentraler Grund fuer die starke Python-Weboekonomie.

# Erklaerung

Ein typischer Ablauf:

1. Server baut `environ` aus der HTTP-Anfrage.
2. Server ruft `application(environ, start_response)` auf.
3. App setzt Status + Header ueber `start_response`.
4. App liefert Body als Bytes-Iterable.
5. Server sendet die fertige Antwort an den Client.

WSGI sorgt also fuer klare Verantwortungsteilung zwischen Server und App-Code.

# Syntax

```python
def application(environ, start_response):
    status = '200 OK'
    headers = [('Content-Type', 'text/plain; charset=utf-8')]
    start_response(status, headers)
    return [b'Hallo WSGI']
```

Achte auf die korrekte Statuszeile und auf `bytes` in der Rueckgabe.

# Merke

- Strings fuer den Body immer mit UTF-8 kodieren.
- Header sind eine Liste von Tupeln.
- Jede Anfrage sollte deterministisch eine gueltige Antwort liefern.

# Parameter

Wichtige `environ`-Schluessel:

- `REQUEST_METHOD`
- `PATH_INFO`
- `QUERY_STRING`
- `SERVER_NAME`
- `SERVER_PORT`

In der Praxis nutzt man Helper-Funktionen, um diese Werte sicher auszulesen.

# Rueckgabewert

Rueckgabe einer WSGI-App:

- iterierbar
- Elemente sind `bytes`

Beispiel:
`return [html_text.encode('utf-8')]`

In [ ]:
# Beispiel 1: Minimale WSGI-App (simuliert)
def app_minimal(environ, start_response):
    status = '200 OK'
    headers = [('Content-Type', 'text/plain; charset=utf-8')]
    start_response(status, headers)
    return ['Hallo aus app_minimal'.encode('utf-8')]

def fake_start_response(status, headers):
    print('STATUS :', status)
    print('HEADERS:', headers)

body = app_minimal({}, fake_start_response)
print('BODY   :', body[0].decode('utf-8'))

# Beispiel 1 - Erklaerung

Dieses Minimalbeispiel zeigt die Pflichtbestandteile ohne Ablenkung.
Sobald diese Form sitzt, kannst du Routing und Business-Logik sauber darauf aufbauen.

In [ ]:
# Beispiel 2: Pfadbasiertes Routing
def app_routing(environ, start_response):
    path = environ.get('PATH_INFO', '/')

    if path == '/':
        status = '200 OK'
        text = 'Startseite'
    elif path == '/kurse':
        status = '200 OK'
        text = 'Kursliste'
    else:
        status = '404 Not Found'
        text = 'Seite nicht gefunden'

    headers = [('Content-Type', 'text/plain; charset=utf-8')]
    start_response(status, headers)
    return [text.encode('utf-8')]

print(app_routing({'PATH_INFO': '/kurse'}, fake_start_response)[0].decode('utf-8'))
print(app_routing({'PATH_INFO': '/x'}, fake_start_response)[0].decode('utf-8'))

# Beispiel 2 - Erklaerung

Ein Router ist im Kern nur eine Entscheidungsstruktur ueber `PATH_INFO`.
Frameworks automatisieren das, aber das Grundprinzip bleibt identisch.

In [ ]:
# Beispiel 3: Mini-Framework mit Route-Registry
class MiniWSGI:
    def __init__(self):
        self.routen = {}

    def route(self, path):
        def decorator(func):
            self.routen[path] = func
            return func
        return decorator

    def __call__(self, environ, start_response):
        path = environ.get('PATH_INFO', '/')
        handler = self.routen.get(path)

        if handler is None:
            status = '404 Not Found'
            body = 'Route nicht gefunden'
        else:
            status = '200 OK'
            body = handler(environ)

        headers = [('Content-Type', 'text/plain; charset=utf-8')]
        start_response(status, headers)
        return [body.encode('utf-8')]

app = MiniWSGI()

@app.route('/')
def startseite(environ):
    return 'Willkommen auf der Startseite'

@app.route('/team')
def teamseite(environ):
    return 'Unser Team stellt sich vor'

print(app({'PATH_INFO': '/'}, fake_start_response)[0].decode('utf-8'))
print(app({'PATH_INFO': '/team'}, fake_start_response)[0].decode('utf-8'))
print(app({'PATH_INFO': '/unknown'}, fake_start_response)[0].decode('utf-8'))

# Beispiel 3 - Erklaerung

Das Mini-Framework zeigt klar:
- Route-Definition ueber Decorator
- Aufruflogik in `__call__`
- saubere Trennung von Routing und Handler-Code

Genau diese Ideen bilden den Kern vieler echter Frameworks.

# Praxisbeispiel

Wir erweitern die App um Query-Handling fuer eine Begruessung.
Dazu lesen wir `QUERY_STRING` und extrahieren den Namen.

In [ ]:
from urllib.parse import parse_qs

@app.route('/hallo')
def hallo(environ):
    query = environ.get('QUERY_STRING', '')
    daten = parse_qs(query, keep_blank_values=True)
    name = daten.get('name', ['Gast'])[0].strip() or 'Gast'
    return f'Hallo {name}, willkommen im WSGI-Training!'

environ = {'PATH_INFO': '/hallo', 'QUERY_STRING': 'name=Esra'}
print(app(environ, fake_start_response)[0].decode('utf-8'))

# Haeufige Fehler

1. Body als `str` statt als `bytes` zurueckgeben.
2. `start_response` vergessen.
3. Keine 404-Route definieren.
4. Query-Parsing ohne Default-Werte.
5. Handler werfen Exceptions ohne saubere Fehlerantwort.

# Best Practice

- Halte den App-Kern klein und erweiterbar.
- Implementiere frueh einen zentralen Fehler-Handler.
- Definiere einheitliche Antwortformate (Text/HTML/JSON).
- Dokumentiere jede Route mit Zweck und erwarteten Parametern.
- Trenne Fachlogik von HTTP-spezifischen Details.

# Tipp

Wenn du dein Mini-Framework besser verstehen willst, zeichne den Ablauf als Sequenz:
Request -> Router -> Handler -> Response.
Dieses Diagramm hilft dir enorm bei Debugging und Erweiterungen.

# Uebung

Erweitere `MiniWSGI` um:

1. Route `/zeit`, die die aktuelle Uhrzeit als Text liefert.
2. Route `/status`, die Serverstatus ausgibt.
3. Eine 500-Fehlerbehandlung, falls ein Handler eine Exception wirft.

In [ ]:
# Loesung
from datetime import datetime

class RobustMiniWSGI(MiniWSGI):
    def __call__(self, environ, start_response):
        path = environ.get('PATH_INFO', '/')
        handler = self.routen.get(path)

        try:
            if handler is None:
                status = '404 Not Found'
                body = 'Route nicht gefunden'
            else:
                status = '200 OK'
                body = handler(environ)
        except Exception:
            status = '500 Internal Server Error'
            body = 'Interner Fehler'

        headers = [('Content-Type', 'text/plain; charset=utf-8')]
        start_response(status, headers)
        return [str(body).encode('utf-8')]

robust_app = RobustMiniWSGI()

@robust_app.route('/zeit')
def zeit_route(environ):
    return datetime.now().strftime('Aktuelle Zeit: %H:%M:%S')

@robust_app.route('/status')
def status_route(environ):
    return 'Serverstatus: OK'

print(robust_app({'PATH_INFO': '/zeit'}, fake_start_response)[0].decode('utf-8'))
print(robust_app({'PATH_INFO': '/status'}, fake_start_response)[0].decode('utf-8'))

# Zusammenfassung

In diesem Notebook hast du WSGI von Grund auf praktisch verstanden:

- Signatur und Rollen von `environ`/`start_response`
- Routing auf Basis von `PATH_INFO`
- Mini-Framework-Denken mit Decorators
- robuste Fehlerbehandlung

Im naechsten Notebook folgt der professionelle Blick: Architektur, Deployment und anspruchsvollere Aufgaben.

# Weiterfuehrende Links

- PEP 3333 (WSGI)
- `wsgiref` in der Python-Standardbibliothek
- Flask intern (WSGI-App als Callable)

## Technischer Tiefgang

Der Fokus liegt auf reproduzierbaren technischen Entscheidungen statt auf isolierten Einzelbeispielen.
Dabei werden Architektur, Robustheit und Betriebsfaehigkeit gemeinsam betrachtet.

## Zentrale Fachbegriffe

HTTP Semantics
Status Code Family
WSGI Callable
Request Lifecycle
Input Sanitization
Header Validation

In [ ]:
# WSGI-Minibeispiel mit Statuscode
def app(environ, start_response):
    path = environ.get("PATH_INFO", "/")
    if path == "/health":
        start_response("200 OK", [("Content-Type", "text/plain")])
        return [b"ok"]
    start_response("404 Not Found", [("Content-Type", "text/plain")])
    return [b"not found"]

## Fallstudie (Praxis)

Waehle ein realistisches Produktionsszenario und beschreibe systematisch Ursache, Risiko und technische Gegenmassnahmen.
Ergaenze mindestens ein Kriterium fuer Monitoring und ein Kriterium fuer Release-Entscheidungen.

## Haeufige Fehler und Debugging-Checkliste

- Ist das Problem reproduzierbar mit klaren Schritten?
- Sind relevante Signale vorhanden (Logs, Tests, Metriken)?
- Wurde eine konkrete Hypothese getestet und falsifiziert/bestaetigt?
- Ist die Korrektur durch einen Regressionstest abgesichert?
- Wurden Betriebsfolgen und Dokumentation mit aktualisiert?

## Pruefungsfragen und Kurzloesungen

1. Warum ist Reproduzierbarkeit in Fehleranalyse und Betrieb zentral?
Kurzloesung: Ohne reproduzierbare Befunde sind Ursachenanalyse, Fix und Absicherung nicht belastbar.
2. Was unterscheidet technische Begriffe von bloessem Buzzword-Einsatz?
Kurzloesung: Praezise Begriffe steuern messbare Entscheidungen und verbessern Teamkommunikation.
3. Welche Mindestkriterien sollte ein Release-Gate enthalten?
Kurzloesung: Teststatus, Sicherheitschecks, Fehlerbudget und nachvollziehbare Freigabeentscheidung.